In [3]:
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier, RandomForestClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, log_loss, RocCurveDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

sns.set_theme(style="whitegrid")

PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"

# ── Output folder: outputs/training_data/ ─────────────────────────────────
BASE_OUT  = PROJECT_ROOT / "outputs" / "training_data"
TABLE_DIR = PROJECT_ROOT / "outputs" / "comparison_tables"

for p in [MODELS_DIR, TABLE_DIR, BASE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Output root:", BASE_OUT)

Output root: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data


In [5]:
# ── Load the 70% training data only ───────────────────────────────────────
# X_test_processed is the held-out 30% — we do NOT load or use it here
X_full_train = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
y_full_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]

# ── Split 70% train → 80% subtrain / 20% validation ──────────────────────
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_full_train, y_full_train,
    test_size=0.20,
    stratify=y_full_train,
    random_state=42
)

# ── Save the new split files so they are clearly named ────────────────────
X_subtrain.to_csv(PROCESSED_DIR / "X_subtrain_80.csv",  index=False)
X_val.to_csv(     PROCESSED_DIR / "X_val_20.csv",        index=False)
y_subtrain.to_csv(PROCESSED_DIR / "y_subtrain_80.csv",  index=False)
y_val.to_csv(     PROCESSED_DIR / "y_val_20.csv",        index=False)

print("Split summary:")
print(f"  X_full_train (70% of original) : {X_full_train.shape}")
print(f"  X_subtrain   (80% of 70% = 56%): {X_subtrain.shape}")
print(f"  X_val        (20% of 70% = 14%): {X_val.shape}")
print(f"\nClass distribution in subtrain:\n{y_subtrain.value_counts()}")
print(f"\nClass distribution in val:\n{y_val.value_counts()}")

# ── CV on subtrain ─────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

Split summary:
  X_full_train (70% of original) : (378, 41)
  X_subtrain   (80% of 70% = 56%): (302, 41)
  X_val        (20% of 70% = 14%): (76, 41)

Class distribution in subtrain:
PCOS
0    203
1     99
Name: count, dtype: int64

Class distribution in val:
PCOS
0    51
1    25
Name: count, dtype: int64


In [6]:
def safe_name(name):
    return (name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", "")
            .replace("+", "plus").replace(":", ""))


def model_dir(model_name):
    """outputs/training_data/<model_name>/"""
    d = BASE_OUT / safe_name(model_name)
    d.mkdir(parents=True, exist_ok=True)
    return d


def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return model.predict(X).astype(float)


def optimize_threshold(y_true, scores):
    best_t, best_v = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        preds = (scores >= t).astype(int)
        v = f1_score(y_true, preds, zero_division=0)
        if v > best_v:
            best_v, best_t = v, t
    return float(best_t)


def metric_block(y_true, scores, threshold):
    preds = (scores >= threshold).astype(int)
    out = {
        "Accuracy"  : accuracy_score(y_true, preds),
        "Error Rate": 1 - accuracy_score(y_true, preds),
        "Precision" : precision_score(y_true, preds, zero_division=0),
        "Recall"    : recall_score(y_true, preds, zero_division=0),
        "F1"        : f1_score(y_true, preds, zero_division=0),
        "F2"        : fbeta_score(y_true, preds, beta=2, zero_division=0),
        "ROC-AUC"   : roc_auc_score(y_true, scores),
        "PR-AUC"    : average_precision_score(y_true, scores),
    }
    try:
        out["Log Loss"] = log_loss(y_true, np.clip(scores, 1e-6, 1 - 1e-6))
    except Exception:
        out["Log Loss"] = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    out.update({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return out, preds

print("Helpers defined.")

Helpers defined.


In [7]:
def plot_confusion(y_true, preds, model_name, out_dir, split_label):
    cm = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Yes"],
                yticklabels=["No", "Yes"], ax=ax)
    ax.set_title(
        f"Training Data ({split_label}) — {model_name}\nConfusion Matrix",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    path = out_dir / "confusion_matrix.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def plot_roc(y_true, scores, model_name, out_dir, split_label):
    auc = roc_auc_score(y_true, scores)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = RocCurveDisplay.from_predictions(y_true, scores, name="", ax=ax)
    disp.line_.set_label(f"Classifier (AUC = {auc:.4f})")
    ax.legend(loc="lower right")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_title(
        f"Training Data ({split_label}) — {model_name}\nROC Curve",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    path = out_dir / "roc_curve.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def plot_stratified_cv(model, model_name, out_dir):
    """10-fold CV on X_subtrain only."""
    fold_acc, fold_f1, fold_auc = [], [], []
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    for tr, va in skf.split(X_subtrain, y_subtrain):
        m = clone(model)
        m.fit(X_subtrain.iloc[tr], y_subtrain.iloc[tr])
        sc    = predict_scores(m, X_subtrain.iloc[va])
        th    = optimize_threshold(y_subtrain.iloc[tr],
                                   predict_scores(m, X_subtrain.iloc[tr]))
        preds = (sc >= th).astype(int)
        fold_acc.append(accuracy_score(y_subtrain.iloc[va], preds))
        fold_f1.append(f1_score(y_subtrain.iloc[va], preds, zero_division=0))
        try:
            fold_auc.append(roc_auc_score(y_subtrain.iloc[va], sc))
        except Exception:
            fold_auc.append(np.nan)

    folds = np.arange(1, 11)
    fig, ax = plt.subplots(figsize=(12, 5))
    for vals, label, marker in [
        (fold_acc, f"Accuracy (mean={np.nanmean(fold_acc):.4f})", "o"),
        (fold_f1,  f"F1       (mean={np.nanmean(fold_f1):.4f})",  "s"),
        (fold_auc, f"ROC-AUC  (mean={np.nanmean(fold_auc):.4f})", "^"),
    ]:
        ax.plot(folds, vals, marker=marker, linewidth=1.8, label=label)
        for x, y in zip(folds, vals):
            ax.annotate(f"{y:.4f}", (x, y),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", fontsize=7.5, color="dimgray")

    ax.set_xticks(folds)
    ax.set_xticklabels([f"Fold {i}" for i in folds])
    ax.set_ylabel("Score")
    ax.set_ylim(max(0, min(fold_acc + fold_f1 + fold_auc) - 0.08), 1.08)
    ax.set_title(
        f"Training Data (Subtrain) — {model_name}\n"
        f"10-Fold Stratified CV ({X_subtrain.shape[1]} Features)",
        fontsize=11, fontweight="bold"
    )
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    path = out_dir / "stratified_cv.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

print("Plot functions defined.")

Plot functions defined.


In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42),

    "SVM": SVC(
        kernel="rbf", probability=True, class_weight="balanced", random_state=42),

    "Gaussian NB": GaussianNB(),

    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42),
        n_estimators=150, random_state=42),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=150, learning_rate=0.5, random_state=42),

    "Gradient Boosting": GradientBoostingClassifier(random_state=42),

    "KNN": KNeighborsClassifier(n_neighbors=21),

    "LDA": LinearDiscriminantAnalysis(),

    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),

    "Perceptron": CalibratedClassifierCV(
        Perceptron(class_weight="balanced", random_state=42), cv=5),
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=42)
else:
    models["XGBoost"] = GradientBoostingClassifier(random_state=43)

models["Stacking ML"] = StackingClassifier(
    estimators=[
        ("rf",  RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ("gb",  GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42),
    cv=5, n_jobs=1,
)

print(f"Total models: {len(models)}")
print(list(models.keys()))

Total models: 14
['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'Gaussian NB', 'Bagging', 'AdaBoost', 'Gradient Boosting', 'KNN', 'LDA', 'QDA', 'Perceptron', 'XGBoost', 'Stacking ML']


In [9]:
results = []

for name, model in models.items():
    print(f"\n▶ Training : {name}")
    out_dir = model_dir(name)

    # ── Fit on SUBTRAIN only ───────────────────────────────────────────────
    t_start = time.time()
    model.fit(X_subtrain, y_subtrain)
    train_time_sec = round(time.time() - t_start, 4)

    # ── Scores ────────────────────────────────────────────────────────────
    subtrain_scores = predict_scores(model, X_subtrain)
    val_scores      = predict_scores(model, X_val)

    # Threshold from subtrain
    threshold = optimize_threshold(y_subtrain, subtrain_scores)

    subtrain_metrics, subtrain_preds = metric_block(y_subtrain, subtrain_scores, threshold)
    val_metrics,      val_preds      = metric_block(y_val,      val_scores,      threshold)

    # ── CV on subtrain ─────────────────────────────────────────────────────
    cv_acc = cross_val_score(clone(model), X_subtrain, y_subtrain,
                             cv=cv, scoring="accuracy", n_jobs=1)
    cv_f1  = cross_val_score(clone(model), X_subtrain, y_subtrain,
                             cv=cv, scoring="f1",       n_jobs=1)
    try:
        cv_auc = cross_val_score(clone(model), X_subtrain, y_subtrain,
                                 cv=cv, scoring="roc_auc", n_jobs=1)
    except Exception:
        cv_auc = np.array([np.nan])

    # ── Plots ──────────────────────────────────────────────────────────────
    # Confusion matrix & ROC on VALIDATION set
    plot_confusion(y_val,   val_preds,   name, out_dir, "Validation 20%")
    plot_roc(y_val,         val_scores,  name, out_dir, "Validation 20%")
    plot_stratified_cv(model,            name, out_dir)

    # ── Save model ─────────────────────────────────────────────────────────
    joblib.dump(model, MODELS_DIR / f"training_data_{safe_name(name)}.joblib")
    (MODELS_DIR / f"training_data_{safe_name(name)}_features.json").write_text(
        json.dumps(list(X_subtrain.columns)))

    # ── Results row ────────────────────────────────────────────────────────
    row = {
        "Model"              : name,
        "train_time_sec"     : train_time_sec,
        "Threshold"          : threshold,
        # CV on subtrain
        "CV Accuracy Mean"   : np.nanmean(cv_acc),
        "CV Accuracy Std"    : np.nanstd(cv_acc),
        "CV F1 Mean"         : np.nanmean(cv_f1),
        "CV F1 Std"          : np.nanstd(cv_f1),
        "CV ROC-AUC Mean"    : np.nanmean(cv_auc),
        "CV ROC-AUC Std"     : np.nanstd(cv_auc),
    }
    # Subtrain metrics (will be very high — expected)
    row.update({f"Subtrain {k}": v for k, v in subtrain_metrics.items()})
    # Validation metrics (key comparison metric)
    row.update({f"Val {k}"     : v for k, v in val_metrics.items()})
    results.append(row)

    print(f"  ✓ Train | Acc={subtrain_metrics['Accuracy']:.4f} "
          f"F1={subtrain_metrics['F1']:.4f} AUC={subtrain_metrics['ROC-AUC']:.4f}")
    print(f"  ✓ Val   | Acc={val_metrics['Accuracy']:.4f} "
          f"F1={val_metrics['F1']:.4f} AUC={val_metrics['ROC-AUC']:.4f} "
          f"Time={train_time_sec:.2f}s")


▶ Training : Logistic Regression
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data\logistic_regression\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data\logistic_regression\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data\logistic_regression\stratified_cv.png
  ✓ Train | Acc=0.9437 F1=0.9154 AUC=0.9785
  ✓ Val   | Acc=0.8684 F1=0.8148 AUC=0.9333 Time=0.01s

▶ Training : Decision Tree
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data\decision_tree\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data\decision_tree\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-c

In [10]:
df_results = pd.DataFrame(results)
df_results["Rank"] = df_results["Val F1"].rank(ascending=False, method="min").astype(int)
df_results = df_results.sort_values(["Rank", "Val ROC-AUC"], ascending=[True, False])

# Save to comparison_tables so NB6 can load it
df_results.to_csv(TABLE_DIR / "training_data_results.csv", index=False)

# Also save a copy inside training_data folder
df_results.to_csv(BASE_OUT / "training_data_results.csv", index=False)

display(df_results[[
    "Rank", "Model",
    "Subtrain Accuracy", "Subtrain F1", "Subtrain ROC-AUC",
    "Val Accuracy",      "Val F1",      "Val ROC-AUC",
    "CV F1 Mean",        "CV ROC-AUC Mean",
    "train_time_sec"
]])

print("\n✓ All done. Outputs saved under:", BASE_OUT)
print("Note: Subtrain metrics will be very high — this is expected.")
print("      Val metrics are the meaningful comparison to test set results.")

,Rank,Model,Subtrain Accuracy,Subtrain F1,Subtrain ROC-AUC,Val Accuracy,Val F1,Val ROC-AUC,CV F1 Mean,CV ROC-AUC Mean,train_time_sec
6,1,AdaBoost,0.963576,0.946860,0.995472,0.907895,0.867925,0.955294,0.760764,0.913733,0.2437
5,2,Bagging,1.000000,1.000000,1.000000,0.881579,0.842105,0.957255,0.790688,0.915598,0.5328
13,3,Stacking ML,1.000000,1.000000,1.000000,0.881579,0.836364,0.967843,0.815046,0.948280,2.8321
2,4,Random Forest,1.000000,1.000000,1.000000,0.868421,0.833333,0.974902,0.801702,0.950631,0.4078
8,4,KNN,0.894040,0.831579,0.954222,0.894737,0.833333,0.947451,0.741242,0.937995,0.0018
12,6,XGBoost,1.000000,1.000000,1.000000,0.868421,0.827586,0.967843,0.790823,0.931206,0.1431
7,6,Gradient Boosting,1.000000,1.000000,1.000000,0.868421,0.827586,0.967843,0.803281,0.929569,0.1774
0,8,Logistic Regression,0.943709,0.915423,0.978454,0.868421,0.814815,0.933333,0.807311,0.914746,0.0126
9,9,LDA,0.933775,0.900000,0.975817,0.868421,0.800000,0.916078,0.802066,0.929656,0.0055
11,10,Perceptron,0.920530,0.882353,0.967906,0.855263,0.792453,0.949804,0.773774,0.910825,0.0433



✓ All done. Outputs saved under: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data
Note: Subtrain metrics will be very high — this is expected.
      Val metrics are the meaningful comparison to test set results.


In [11]:
# CELL 8 — Overfitting Gap Analysis
df_gap = df_results[["Model", "Subtrain F1", "Val F1",
                      "Subtrain ROC-AUC", "Val ROC-AUC",
                      "Subtrain Accuracy", "Val Accuracy"]].copy()

df_gap["F1 Gap"]       = (df_gap["Subtrain F1"]       - df_gap["Val F1"]).round(4)
df_gap["AUC Gap"]      = (df_gap["Subtrain ROC-AUC"]  - df_gap["Val ROC-AUC"]).round(4)
df_gap["Acc Gap"]      = (df_gap["Subtrain Accuracy"] - df_gap["Val Accuracy"]).round(4)

def overfit_label(gap):
    if gap >= 0.15:   return "Severe Overfit"
    elif gap >= 0.08: return "Moderate Overfit"
    elif gap >= 0.03: return "Mild Overfit"
    else:             return "Healthy"

df_gap["Overfit Status"] = df_gap["F1 Gap"].apply(overfit_label)
df_gap = df_gap.sort_values("F1 Gap", ascending=False)

df_gap.to_csv(BASE_OUT / "overfitting_gap_analysis.csv", index=False)
df_gap.to_csv(TABLE_DIR / "overfitting_gap_analysis.csv", index=False)
display(df_gap)

# ── Plot the gap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = df_gap["F1 Gap"].apply(
    lambda g: "#d62728" if g >= 0.15 else
              "#ff7f0e" if g >= 0.08 else
              "#ffbb78" if g >= 0.03 else "#2ca02c"
)
bars = ax.bar(df_gap["Model"], df_gap["F1 Gap"],
              color=colors, edgecolor="white", alpha=0.9)

for bar, val, status in zip(bars, df_gap["F1 Gap"], df_gap["Overfit Status"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f"{val:.4f}\n{status}",
            ha="center", va="bottom", fontsize=7.5, fontweight="bold")

ax.axhline(0.15, color="#d62728", linestyle="--", lw=1.2, label="Severe overfit threshold (0.15)")
ax.axhline(0.08, color="#ff7f0e", linestyle="--", lw=1.2, label="Moderate overfit threshold (0.08)")
ax.axhline(0.03, color="#ffbb78", linestyle="--", lw=1.2, label="Mild overfit threshold (0.03)")
ax.set_xticklabels(df_gap["Model"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("F1 Gap (Subtrain − Val)", fontsize=10)
ax.set_title(
    "Overfitting Analysis — F1 Gap per Model\n"
    "(Subtrain F1 minus Validation F1)",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9)
plt.tight_layout()
path = BASE_OUT / "overfitting_gap_analysis.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {path}")

,Model,Subtrain F1,Val F1,Subtrain ROC-AUC,Val ROC-AUC,Subtrain Accuracy,Val Accuracy,F1 Gap,AUC Gap,Acc Gap,Overfit Status
1,Decision Tree,1.000000,0.730769,1.000000,0.801569,1.000000,0.815789,0.2692,0.1984,0.1842,Severe Overfit
3,SVM,0.964103,0.697674,0.996865,0.942745,0.976821,0.828947,0.2664,0.0541,0.1479,Severe Overfit
10,QDA,0.970000,0.775510,0.998209,0.924706,0.980132,0.855263,0.1945,0.0735,0.1249,Severe Overfit
7,Gradient Boosting,1.000000,0.827586,1.000000,0.967843,1.000000,0.868421,0.1724,0.0322,0.1316,Severe Overfit
12,XGBoost,1.000000,0.827586,1.000000,0.967843,1.000000,0.868421,0.1724,0.0322,0.1316,Severe Overfit
2,Random Forest,1.000000,0.833333,1.000000,0.974902,1.000000,0.868421,0.1667,0.0251,0.1316,Severe Overfit
13,Stacking ML,1.000000,0.836364,1.000000,0.967843,1.000000,0.881579,0.1636,0.0322,0.1184,Severe Overfit
5,Bagging,1.000000,0.842105,1.000000,0.957255,1.000000,0.881579,0.1579,0.0427,0.1184,Severe Overfit
0,Logistic Regression,0.915423,0.814815,0.978454,0.933333,0.943709,0.868421,0.1006,0.0451,0.0753,Moderate Overfit
9,LDA,0.900000,0.800000,0.975817,0.916078,0.933775,0.868421,0.1000,0.0597,0.0654,Moderate Overfit


Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\training_data\overfitting_gap_analysis.png
